# R.O.A.D. Barbados — Tier 1 Kraken training (Colab)

Self-contained: run cells top to bottom, no need to run `bootstrap.ipynb` first. Fine-tunes the CATMuS pretrained Kraken model (`src/kraken/train.py`) on `data/train_split.csv`, validates on `data/val_split.csv` (the same stratified splits every tier uses — see `src/data/make_splits.py`), then runs `src/kraken/infer.py` — the pageseg-bug-fixed direct line recognizer — on both `val_split.csv` (local scoring) and `Test.csv` (real submission).

Requires the same one-time `RCLONE_CONF` Colab secret as `bootstrap.ipynb` — see `runners/colab/README.md` if you haven't set that up. Code, data, and this notebook's outputs (checkpoints, submissions) live on Drive B; do not print the secret or commit `rclone.conf`.

## 1. Check GPU

In [ ]:
import subprocess

gpu_info = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
)
if gpu_info.returncode != 0 or not gpu_info.stdout.strip():
    print("No GPU detected — Kraken's CRNN is small enough to train on CPU if you have to, just slower.")
    print("Runtime > Change runtime type > GPU is still recommended.")
else:
    print(gpu_info.stdout.strip())

## 2. Install rclone and restore `rclone.conf` from the Colab secret

In [ ]:
!apt-get -qq update && apt-get -qq install -y rclone
!rclone version | head -1

In [ ]:
import os
from pathlib import Path
from google.colab import userdata

conf_text = userdata.get("RCLONE_CONF")
conf_dir = Path.home() / ".config" / "rclone"
conf_dir.mkdir(parents=True, exist_ok=True)
conf_path = conf_dir / "rclone.conf"
conf_path.write_text(conf_text)
os.chmod(conf_path, 0o600)
del conf_text  # never print/log this
print(f"rclone.conf restored ({conf_path.stat().st_size} bytes)")

In [ ]:
REMOTE = "roadB"
DRIVE_ROOT = f"{REMOTE}:road-barbados-htr"

check = subprocess.run(["rclone", "lsd", f"{REMOTE}:"], capture_output=True, text=True)
if check.returncode != 0:
    raise RuntimeError(
        "rclone can't reach the 'roadB' remote — check the RCLONE_CONF secret is set and "
        "enabled for this notebook. stderr:\n" + check.stderr
    )
print("rclone -> Drive B connectivity OK")

## 3. Local paths + pull code from Drive

In [ ]:
TIER = "tier1_kraken"

CODE_DIR = "/content/road-barbados-htr"
DATA_DIR = "/content/data"
MODEL_CACHE_DIR = "/content/model_cache"
CKPT_DIR = f"/content/checkpoints/{TIER}"
LOG_DIR = f"/content/logs/{TIER}"
CKPT_REMOTE = f"{DRIVE_ROOT}/experiments/{TIER}/checkpoints"
LOG_REMOTE = f"{DRIVE_ROOT}/experiments/{TIER}/logs"
SUBMISSIONS_REMOTE = f"{DRIVE_ROOT}/submissions"

for d in (CODE_DIR, DATA_DIR, MODEL_CACHE_DIR, CKPT_DIR, LOG_DIR):
    os.makedirs(d, exist_ok=True)

print(f"TIER={TIER}")

In [ ]:
!rclone copy "{DRIVE_ROOT}" "{CODE_DIR}" -P \
  --exclude "data/images/**" \
  --exclude "data/*.csv" \
  --exclude "model_cache/**" \
  --exclude "experiments/**" \
  --exclude "submissions/**" \
  --exclude ".git/**" \
  --exclude ".venv-*/**"

## 4. Install Kraken (fresh each session)

In [ ]:
!pip install -q -r "{CODE_DIR}/src/kraken/requirements.txt"

import torch
print("torch", torch.__version__, "cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("WARNING: no CUDA — training will run on CPU and be much slower. "
          "Check Runtime > Change runtime type, or reinstall torch from a CUDA index if this is unexpected on a GPU runtime.")

## 5. Pull data to local disk

Same reasoning as `bootstrap.ipynb`: always train/infer against local Colab disk, never the `roadB:` remote directly.

In [ ]:
!rclone copy "{DRIVE_ROOT}/data/images" "{DATA_DIR}/images" -P
!rclone copy "{DRIVE_ROOT}/data" "{DATA_DIR}" -P --include "*.csv" --max-depth 1

## 6. Get the CATMuS pretrained model

Pulls a previously-cached copy from Drive B if one exists; otherwise fetches it fresh. Tries `kraken get` first, falls back to a direct Zenodo API download if that fails — this exact fallback was needed when testing this pipeline on Windows (a `htrmopo` encoding bug unrelated to Colab), included here defensively since it's cheap insurance and Zenodo's plain HTTP API works everywhere.

In [ ]:
CATMUS_PATH = f"{MODEL_CACHE_DIR}/catmus_pretrained.mlmodel"
CATMUS_DOI = "10.5281/zenodo.10592716"

!rclone copy "{DRIVE_ROOT}/model_cache" "{MODEL_CACHE_DIR}" -P

if not os.path.exists(CATMUS_PATH):
    print("Not cached on Drive — fetching fresh")
    got_it = False
    try:
        result = subprocess.run(["kraken", "get", CATMUS_DOI], capture_output=True, text=True, check=True)
        print(result.stdout)
        import glob
        candidates = glob.glob(os.path.expanduser("~/.local/share/kraken/**/*.mlmodel"), recursive=True)
        if candidates:
            import shutil
            shutil.copy2(candidates[0], CATMUS_PATH)
            got_it = True
    except Exception as exc:
        print(f"`kraken get` failed ({exc}), falling back to direct Zenodo download")

    if not got_it:
        import requests
        meta = requests.get("https://zenodo.org/api/records/10592716").json()
        mlmodel_file = next(f for f in meta["files"] if f["key"].endswith(".mlmodel"))
        url = mlmodel_file["links"]["self"]
        print(f"downloading {mlmodel_file['key']} ({mlmodel_file['size']} bytes) from Zenodo directly")
        with requests.get(url, stream=True) as r:
            r.raise_for_status()
            with open(CATMUS_PATH, "wb") as f:
                for chunk in r.iter_content(chunk_size=1 << 20):
                    f.write(chunk)

    subprocess.run(["rclone", "copy", MODEL_CACHE_DIR, f"{DRIVE_ROOT}/model_cache", "-P"], check=True)
    print("cached CATMuS model to Drive B for future sessions")
else:
    print(f"using cached {CATMUS_PATH}")

## 7. Checkpoint detect-and-resume

Kraken (`ketos train`) saves a new `.mlmodel` per epoch, e.g. `kraken_model_3.mlmodel` — not the `checkpoint-N/` directory convention HF Trainer uses in `bootstrap.ipynb`. This pulls any existing checkpoints down and finds the highest epoch number.

In [ ]:
import re
from pathlib import Path

!rclone copy "{CKPT_REMOTE}" "{CKPT_DIR}" -P

def find_latest_kraken_checkpoint(ckpt_dir):
    candidates = sorted(
        Path(ckpt_dir).glob("*.mlmodel"),
        key=lambda p: int(m.group(1)) if (m := re.search(r"_(\d+)\.mlmodel$", p.name)) else -1,
    )
    return str(candidates[-1]) if candidates else None

LATEST_CKPT = find_latest_kraken_checkpoint(CKPT_DIR)
if LATEST_CKPT:
    LOAD_MODEL, LOAD_HYPER_PARAMS = LATEST_CKPT, True
    print(f"Resuming from {LOAD_MODEL}")
else:
    LOAD_MODEL, LOAD_HYPER_PARAMS = CATMUS_PATH, False
    print(f"No checkpoint found — starting fresh from {LOAD_MODEL}")

## 8. Periodic checkpoint push-back

`ketos train` runs as a subprocess (not a Python training loop we can hook a callback into), so this runs a background thread that `rclone copy`'s `CKPT_DIR`/`LOG_DIR` to Drive every couple of minutes while training runs — same reasoning as the VLM tier's `RcloneCheckpointSyncCallback`: Colab session/idle timeouts can happen with no warning.

In [ ]:
import threading
import time

_stop_sync = threading.Event()

def _sync_loop(interval_seconds=120):
    while not _stop_sync.wait(interval_seconds):
        subprocess.run(["rclone", "copy", CKPT_DIR, CKPT_REMOTE, "-P"], capture_output=True)
        subprocess.run(["rclone", "copy", LOG_DIR, LOG_REMOTE, "-P"], capture_output=True)
        print(f"[sync] checkpoints mirrored to Drive B at {time.strftime('%H:%M:%S')}")

def start_checkpoint_sync():
    _stop_sync.clear()
    t = threading.Thread(target=_sync_loop, daemon=True)
    t.start()
    return t

def stop_checkpoint_sync_and_final_push():
    _stop_sync.set()
    subprocess.run(["rclone", "copy", CKPT_DIR, CKPT_REMOTE, "-P"], check=True)
    subprocess.run(["rclone", "copy", LOG_DIR, LOG_REMOTE, "-P"], check=True)
    print("final checkpoint sync done")

print("checkpoint sync helpers ready")

## 9. Train

Calls `src/kraken/train.py` exactly as `runners/narval/submit_kraken_train.sb` does — same script, same CLI contract, only the environment differs. Trains on `train_split.csv`, validates on `val_split.csv` explicitly (`ketos train -e`, not a random `-p` resplit) so this is comparable to every other tier.

In [ ]:
sync_thread = start_checkpoint_sync()

resume_flags = f'--load_model "{LOAD_MODEL}"' + (" --load_hyper_parameters" if LOAD_HYPER_PARAMS else "")

!cd "{CODE_DIR}" && python src/kraken/train.py \
    --config src/kraken/config.yaml \
    --train_csv "{DATA_DIR}/train_split.csv" \
    --val_csv "{DATA_DIR}/val_split.csv" \
    --base_image_dir "{DATA_DIR}/images" \
    --output "{CKPT_DIR}/kraken_model" \
    --log_dir "{LOG_DIR}" \
    {resume_flags}

stop_checkpoint_sync_and_final_push()

## 10. Score on val_split, then run the real Test.csv submission

Uses `src/kraken/infer.py` — the pageseg-bug-fixed direct line recognizer — against the best/latest checkpoint from training. Adjust `--lm_rescore` on if you want to try the lexicon+bigram rescoring pass (`src/kraken/lm_rescore.py`); it's opt-in and not yet validated against a trained model, so compare scores with/without it before trusting it by default.

In [ ]:
BEST_CKPT = find_latest_kraken_checkpoint(CKPT_DIR)
print(f"scoring with {BEST_CKPT}")

!cd "{CODE_DIR}" && python src/kraken/infer.py \
    --model_path "{BEST_CKPT}" \
    --input_csv "{DATA_DIR}/val_split.csv" \
    --base_image_dir "{DATA_DIR}/images" \
    --output_csv "/content/tier1_val_preds.csv"

!cd "{CODE_DIR}" && python -m src.metrics.weighted_wer_cer \
    --gt "{DATA_DIR}/val_split.csv" \
    --pred "/content/tier1_val_preds.csv" \
    --save-report "/content/tier1_val_report.csv"

In [ ]:
import time as _time
TIMESTAMP = _time.strftime("%Y%m%d-%H%M%S")
SUBMISSION_PATH = f"/content/tier1_kraken_{TIMESTAMP}.csv"

!cd "{CODE_DIR}" && python src/kraken/infer.py \
    --model_path "{BEST_CKPT}" \
    --input_csv "{DATA_DIR}/Test.csv" \
    --base_image_dir "{DATA_DIR}/images" \
    --output_csv "{SUBMISSION_PATH}"

!cd "{CODE_DIR}" && python src/vlm/validate_submission.py \
    --submission "{SUBMISSION_PATH}" \
    --sample_submission "{DATA_DIR}/SampleSubmission.csv"

## 11. Push everything back to Drive B

In [ ]:
subprocess.run(["rclone", "copy", "/content/tier1_val_preds.csv", f"{DRIVE_ROOT}/experiments/{TIER}/", "-P"], check=True)
subprocess.run(["rclone", "copy", "/content/tier1_val_report.csv", f"{DRIVE_ROOT}/experiments/{TIER}/", "-P"], check=True)
subprocess.run(["rclone", "copy", SUBMISSION_PATH, SUBMISSIONS_REMOTE, "-P"], check=True)
print("pushed val predictions, scoring report, and submission to Drive B")
print(f"submission filename: {os.path.basename(SUBMISSION_PATH)}")
print("pull it down with: rclone copy \"" + SUBMISSIONS_REMOTE + "/" + os.path.basename(SUBMISSION_PATH) + "\" . -P")